In [2]:
# Aim: Initialize the data, validation protocol, feature definitions, and required libraries for the target-weighted CatBoost experiment.

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor


RANDOM_STATE = 42

# Load prepared datasets
train_final = pd.read_csv("../data/train_final.csv")
test_final = pd.read_csv("../data/test_final.csv")

target = "total_sales"

# Final predictor set
features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

numeric_features = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

X = train_final[features].copy()
y = train_final[target].copy()

X_test = test_final[features].copy()

# Locked validation protocol
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)
print("Target shape:", y.shape)
print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))
print("Validation folds:", kf.get_n_splits())

Train shape: (6818, 11)
Test shape: (1705, 11)
Target shape: (6818,)
Categorical features: 7
Numeric features: 4
Validation folds: 5


In [3]:
# Aim: Test mild target-magnitude weighting in CatBoost to determine whether emphasizing higher-sales observations improves RMSE.

alpha = 0.10

weighted_oof = np.zeros(len(y))
weighted_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    sample_weights = (
        y_tr / y_tr.median()
    ) ** alpha

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=categorical_features,
        sample_weight=sample_weights
    )

    pred = model.predict(X_val)

    weighted_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    weighted_fold_rmse.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print(f"\nAlpha: {alpha}")
print(f"Mean fold RMSE: {np.mean(weighted_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, weighted_oof)):.6f}")

Fold 1: 1065.702698
Fold 2: 1053.967626
Fold 3: 1090.484200
Fold 4: 1110.497926
Fold 5: 1062.726746

Alpha: 0.1
Mean fold RMSE: 1076.675839
OOF RMSE: 1076.873979
